# Lesson 7: wav2vec 2.0 IPA 识别教程

本教程演示如何使用预训练的 wav2vec 2.0 模型进行 IPA（国际音标）识别。

## 学习目标
- 理解自监督学习和预训练-微调范式
- 学习使用 HuggingFace Transformers
- 掌握 wav2vec 2.0 微调流程
- 理解 CTC 解码原理
- 计算 WER/CER 指标

## 1. 环境准备

In [ ]:
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor
)

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print("环境准备完成！")

## 2. wav2vec 2.0 原理

### 自监督学习
wav2vec 2.0 使用自监督学习在大量无标注音频上预训练：
1. **掩码预测**: 随机掩盖部分音频，预测被掩盖的部分
2. **对比学习**: 学习区分真实语音和负样本
3. **量化**: 将连续的音频特征离散化

### 模型架构
```
原始音频 (16kHz)
    ↓
CNN 特征提取器 (7层)
    ↓
Transformer 编码器 (12层)
    ↓
CTC 解码器
    ↓
IPA 音素序列
```

### 预训练-微调范式
- **预训练**: 在 Librispeech (960h) 等大规模数据上学习通用表示
- **微调**: 在小规模标注数据上微调 CTC 头，适应特定任务

## 3. 加载预训练模型

In [ ]:
# 模型名称
model_name = "facebook/wav2vec2-base"

# 加载特征提取器
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

print(f"采样率: {feature_extractor.sampling_rate} Hz")
print(f"特征提取器配置:")
print(f"  - 特征大小: {feature_extractor.feature_size}")
print(f"  - 归一化: {feature_extractor.do_normalize}")

## 4. 准备数据

假设我们有以下格式的数据：
```csv
audio_path,ipa_text
audio1.wav,h ə l oʊ
audio2.wav,w ɜː l d
```

In [ ]:
# 示例数据
data_file = project_root / 'material' / 'lesson_7' / 'data.csv'

if data_file.exists():
    df = pd.read_csv(data_file)
    print(f"加载数据: {len(df)} 个样本")
    print(f"\n前 5 行:")
    display(df.head())
    
    # 统计音素
    all_phones = set()
    for text in df['ipa_text']:
        phones = text.split()
        all_phones.update(phones)
    
    print(f"\n音素数量: {len(all_phones)}")
    print(f"音素列表: {sorted(all_phones)[:20]}...")  # 显示前 20 个
else:
    print(f"数据文件不存在: {data_file}")
    print("\n创建示例数据...")
    
    # 创建示例数据
    sample_data = {
        'audio_path': ['audio1.wav', 'audio2.wav', 'audio3.wav'],
        'ipa_text': ['h ə l oʊ', 'w ɜː l d', 'ɡ ʊ d']
    }
    df = pd.DataFrame(sample_data)
    display(df)

## 5. 创建词汇表

从训练数据中提取所有唯一的音素，创建词汇表。

In [ ]:
def extract_vocab(texts):
    """从文本中提取词汇表"""
    vocab = set()
    for text in texts:
        phones = text.split()
        vocab.update(phones)
    
    # 添加特殊标记
    vocab = sorted(list(vocab))
    vocab_dict = {v: k for k, v in enumerate(vocab)}
    
    # 添加 CTC blank token
    vocab_dict['[PAD]'] = len(vocab_dict)
    vocab_dict['[UNK]'] = len(vocab_dict)
    vocab_dict['[CTC]'] = len(vocab_dict)
    
    return vocab_dict

# 创建词汇表
if 'df' in locals() and len(df) > 0:
    vocab_dict = extract_vocab(df['ipa_text'])
    print(f"词汇表大小: {len(vocab_dict)}")
    print(f"\n词汇表示例:")
    for i, (k, v) in enumerate(list(vocab_dict.items())[:10]):
        print(f"  {k}: {v}")

## 6. 创建 Tokenizer 和 Processor

In [ ]:
if 'vocab_dict' in locals():
    # 保存词汇表
    import json
    vocab_file = project_root / 'results' / 'vocab.json'
    vocab_file.parent.mkdir(exist_ok=True, parents=True)
    
    with open(vocab_file, 'w', encoding='utf-8') as f:
        json.dump(vocab_dict, f, ensure_ascii=False, indent=2)
    
    # 创建 tokenizer
    tokenizer = Wav2Vec2CTCTokenizer(
        str(vocab_file),
        unk_token='[UNK]',
        pad_token='[PAD]',
        word_delimiter_token=' '
    )
    
    # 创建 processor
    processor = Wav2Vec2Processor(
        feature_extractor=feature_extractor,
        tokenizer=tokenizer
    )
    
    print("Tokenizer 和 Processor 创建完成！")
    print(f"词汇表大小: {len(tokenizer)}")

## 7. 数据预处理

将音频和文本转换为模型输入格式。

In [ ]:
def prepare_dataset(batch):
    """准备数据集"""
    # 加载音频
    audio_path = batch['audio_path']
    speech_array, sampling_rate = torchaudio.load(audio_path)
    
    # 重采样到 16kHz
    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
        speech_array = resampler(speech_array)
    
    # 转换为单声道
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0, keepdim=True)
    
    speech_array = speech_array.squeeze().numpy()
    
    # 处理音频
    batch['input_values'] = processor(
        speech_array,
        sampling_rate=16000
    ).input_values[0]
    
    # 处理文本
    with processor.as_target_processor():
        batch['labels'] = processor(batch['ipa_text']).input_ids
    
    return batch

print("数据预处理函数定义完成！")

## 8. 加载微调后的模型（如果存在）

In [ ]:
model_path = project_root / 'checkpoints' / 'wav2vec_ipa'

if model_path.exists():
    print(f"加载微调后的模型: {model_path}")
    model = Wav2Vec2ForCTC.from_pretrained(str(model_path))
    processor = Wav2Vec2Processor.from_pretrained(str(model_path))
    print("模型加载完成！")
else:
    print(f"微调后的模型不存在: {model_path}")
    print("\n加载预训练模型...")
    
    if 'vocab_dict' in locals():
        model = Wav2Vec2ForCTC.from_pretrained(
            model_name,
            vocab_size=len(vocab_dict),
            pad_token_id=processor.tokenizer.pad_token_id,
            ctc_loss_reduction='mean'
        )
        print("预训练模型加载完成！")
    else:
        print("需要先创建词汇表")

# 移动到设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'model' in locals():
    model = model.to(device)
    print(f"模型已移动到: {device}")

## 9. 推理演示

使用模型进行 IPA 识别。

In [ ]:
def transcribe(audio_path, model, processor, device):
    """转录音频为 IPA"""
    # 加载音频
    speech_array, sampling_rate = torchaudio.load(audio_path)
    
    # 重采样
    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
        speech_array = resampler(speech_array)
    
    # 转换为单声道
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0, keepdim=True)
    
    speech_array = speech_array.squeeze().numpy()
    
    # 处理音频
    inputs = processor(
        speech_array,
        sampling_rate=16000,
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 推理
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # CTC 解码
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    
    return transcription

# 示例推理
if 'model' in locals() and 'df' in locals() and len(df) > 0:
    sample_audio = df.iloc[0]['audio_path']
    sample_text = df.iloc[0]['ipa_text']
    
    if Path(sample_audio).exists():
        print(f"音频文件: {sample_audio}")
        print(f"真实标注: {sample_text}")
        
        prediction = transcribe(sample_audio, model, processor, device)
        print(f"模型预测: {prediction}")
    else:
        print(f"音频文件不存在: {sample_audio}")
        print("\n请先运行训练脚本生成模型：")
        print("python scripts/lesson_07_wav2vec_ipa.py --mode train ...")

## 10. CTC 解码原理

CTC (Connectionist Temporal Classification) 解码过程：

1. **模型输出**: 每个时间步的音素概率分布
2. **贪婪解码**: 选择每个时间步概率最高的音素
3. **去重**: 移除连续重复的音素
4. **移除 blank**: 移除 CTC blank token

示例：
```
模型输出: h h [CTC] ə ə l l l oʊ [CTC]
去重后:   h [CTC] ə l oʊ [CTC]
最终结果: h ə l oʊ
```

In [ ]:
# 可视化 CTC 解码过程
if 'model' in locals() and 'df' in locals() and len(df) > 0:
    sample_audio = df.iloc[0]['audio_path']
    
    if Path(sample_audio).exists():
        # 加载音频
        speech_array, sampling_rate = torchaudio.load(sample_audio)
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
            speech_array = resampler(speech_array)
        if speech_array.shape[0] > 1:
            speech_array = torch.mean(speech_array, dim=0, keepdim=True)
        speech_array = speech_array.squeeze().numpy()
        
        # 处理
        inputs = processor(speech_array, sampling_rate=16000, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 推理
        model.eval()
        with torch.no_grad():
            logits = model(**inputs).logits
        
        # 获取概率
        probs = torch.nn.functional.softmax(logits, dim=-1)
        predicted_ids = torch.argmax(logits, dim=-1)[0]
        
        # 可视化前 50 个时间步
        plt.figure(figsize=(15, 6))
        plt.imshow(probs[0, :50, :].cpu().T, aspect='auto', origin='lower', cmap='viridis')
        plt.colorbar(label='概率')
        plt.xlabel('时间步', fontsize=12)
        plt.ylabel('音素 ID', fontsize=12)
        plt.title('CTC 输出概率分布（前 50 个时间步）', fontsize=14)
        plt.tight_layout()
        plt.show()
        
        # 显示预测序列
        print("\n预测的音素 ID 序列（前 50 个时间步）:")
        print(predicted_ids[:50].cpu().numpy())

## 11. 评估指标

### WER (Word Error Rate)
$$WER = \frac{S + D + I}{N}$$

其中：
- S: 替换错误数
- D: 删除错误数
- I: 插入错误数
- N: 参考文本中的词数

### CER (Character Error Rate)
与 WER 类似，但在字符级别计算。

### PER (Phoneme Error Rate)
在音素级别计算错误率。

In [ ]:
def calculate_wer(reference, hypothesis):
    """计算 WER"""
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    
    # 使用编辑距离
    d = np.zeros((len(ref_words) + 1, len(hyp_words) + 1))
    
    for i in range(len(ref_words) + 1):
        d[i][0] = i
    for j in range(len(hyp_words) + 1):
        d[0][j] = j
    
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            if ref_words[i-1] == hyp_words[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                substitution = d[i-1][j-1] + 1
                insertion = d[i][j-1] + 1
                deletion = d[i-1][j] + 1
                d[i][j] = min(substitution, insertion, deletion)
    
    wer = d[len(ref_words)][len(hyp_words)] / len(ref_words)
    return wer

# 示例
reference = "h ə l oʊ w ɜː l d"
hypothesis = "h ə l oʊ w ɜː l"

wer = calculate_wer(reference, hypothesis)
print(f"参考文本: {reference}")
print(f"预测文本: {hypothesis}")
print(f"WER: {wer:.2%}")

## 12. 总结

本教程演示了：
1. ✅ wav2vec 2.0 的原理和架构
2. ✅ 加载预训练模型
3. ✅ 创建词汇表和 Tokenizer
4. ✅ 数据预处理流程
5. ✅ 模型推理和 CTC 解码
6. ✅ 评估指标计算（WER/CER）

### 关键技术点
- **自监督学习**: 在无标注数据上学习通用表示
- **预训练-微调**: 迁移学习范式
- **CTC 解码**: 处理不对齐的序列
- **HuggingFace Transformers**: 简化模型使用

### 训练模型
要训练自己的模型，运行：
```bash
python scripts/lesson_07_wav2vec_ipa.py --mode train \
    --csv_file material/lesson_7/data.csv \
    --model_name facebook/wav2vec2-base \
    --output_dir checkpoints/wav2vec_ipa \
    --epochs 10
```

### 下一步
- 尝试不同的预训练模型（wav2vec2-large, xlsr-53）
- 调整超参数（学习率、batch size）
- 使用数据增强提高泛化能力
- 尝试 beam search 解码